In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%matplotlib inline

import pandas as pd
import matplotlib.pyplot as plt

from quick_pp.database.objects import Project
from quick_pp.database.db_connector import DBConnector

db_conn = DBConnector()

# Load well from saved file
project_name = "30-7a"
well_name = "30-7a-2"
with db_conn.get_session() as db_session:
    project = Project(db_session, name=project_name)
    df = project.get_all_data()
    well_data = project.get_well_data(well_name)

***
## Leverett J Method using FZI Rock Types

In [ ]:
import numpy as np

32 * np.cos(np.radians(0))

In [ ]:
from quick_pp.rock_type import calc_fzi, rock_typing, calc_r35, plot_fzi, plot_winland
from quick_pp.core_calibration import (
    fit_j_curve,
    j_xplot,
    leverett_j,
    sw_shf_leverett_j,
    poroperm_xplot,
    pc_xplot,
)

from utils import restructure_scal_data, string_to_int_hash

raw_core_data = pd.read_excel(rf"data\{project_name}_SCAL.xlsx")
core_data = restructure_scal_data(raw_core_data)
core_data["Sample"] = core_data["Sample ID"]
core_data["SampleID"] = core_data["Sample ID"].apply(string_to_int_hash)
core_data["CPORE"] = core_data["PHI_frac"]
core_data["CPERM"] = core_data["K_mD"]
core_data["PC"] = core_data["Pc"]
core_data["PC_RES"] = core_data["PC"] * (26 / 72)
core_data["SW"] = core_data["Sw"]
core_data["SWN"] = core_data.groupby("Sample")["SW"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min())
)

# Calculate J
ift = 26
theta = 0

core_data["J"] = leverett_j(
    core_data["PC_RES"], ift, theta, core_data["CPERM"], core_data["CPORE"]
)

# Filter data
# conditions = (
#     (core_data["K mD"] > 0) & (core_data["Class"] == "Good")
#     # & (core_data['PC'] <= 40)
# )
# core_data = core_data[conditions].copy()
core_data.drop_duplicates(subset=["CPORE", "CPERM", "SW"], keep="last", inplace=True)

# Plot J
j_xplot(core_data["SWN"], core_data["J"], ylim=(0, 15))

In [ ]:
import json
from quick_pp.rock_type import plot_fzi

# Load FZI cutoffs
with open(rf"data\04_project\{project_name}\outputs\fzi_cutoffs.json", "rb") as file:
    fzi_cutoffs = json.load(file)

# FZI
fzi = calc_fzi(core_data["CPORE"], core_data["CPERM"])
rock_flag = rock_typing(fzi, fzi_cutoffs, higher_is_better=True)
core_data["ROCK_FLAG"] = rock_flag

plot_fzi(
    core_data["CPORE"], core_data["CPERM"], rock_type=rock_flag, cut_offs=fzi_cutoffs
)
print(pd.Series(rock_flag).value_counts().sort_index())

In [ ]:
from utils import plot_pc_by_prt

plot_pc_by_prt(core_data, 80)

In [ ]:
# Plot PTSD distribution
from utils import plot_ptsd_by_prt

copy_df = plot_ptsd_by_prt(core_data, ift, theta)

In [ ]:
from ipywidgets import interact, widgets
import plotly.graph_objects as go

from quick_pp.core_calibration import pc_xplot, poroperm_xplot, pc_xplot_plotly

rock_flag_widget = widgets.SelectMultiple(
    options=["All"] + sorted(list(core_data["ROCK_FLAG"].unique())),
    value=["All"],
    description="Rock Flag:",
)


@interact(rock_flag=rock_flag_widget)
def param(rock_flag):
    # Plot all data on poroperm plot
    poroperm_xplot(core_data["CPORE"], core_data["CPERM"])
    data = (
        core_data[core_data.ROCK_FLAG.isin(rock_flag)]
        if any([l for l in rock_flag if l != "All"])
        else core_data
    )

    # Plot filtered data
    poroperm_data = data.drop_duplicates(subset=["CPORE", "CPERM"], keep="last")
    poroperm_xplot(
        poroperm_data["CPORE"],
        poroperm_data["CPERM"],
        core_group=poroperm_data["SampleID"],
    )
    plt.show()

    fig = go.Figure()
    for label, temp_df in data.groupby("SampleID"):
        fig = pc_xplot_plotly(
            temp_df["SWN"], temp_df["PC_RES"], label=label, ylim=(0, 1.5), fig=fig
        )
    fig.show()


plt.close("all")

#### QC the Pc data

The capillary pressure measurements for each Sample are plotted on a log-log plot.
The data points should fall on a relatively straight line indicating good data quality.

Based 
select the dataset for each rock type
curve fitting

In [ ]:
from ipywidgets import interact, widgets

from quick_pp.core_calibration import fit_j_curve

prt = widgets.Dropdown(
    options=sorted(core_data["ROCK_FLAG"].unique()), description="PRT:"
)

conditions = (core_data["SWN"] > 0.1) & (core_data["SWN"] < 0.9)
filtered_data = core_data[conditions].copy()


@interact(prt=prt)
def param(prt):
    data = filtered_data[filtered_data["ROCK_FLAG"] == prt]
    a, b = fit_j_curve(data["SWN"], data["J"])
    j_xplot(
        data["SWN"],
        data["J"],
        a=a,
        b=b,
        label=f"Sample {sample}: a:{a}, b:{b}",
        core_group=data["Sample"],
        log_log=True,
        ylim=(0, 0.5),
    )

In [ ]:
from sklearn.metrics import root_mean_squared_error

excluded_samples = [
    72,
    77,
    81,
    51,
    38,
    54,
    26,
    66,
    43,
    69,
    55,
    57,
    75,
    # 50, 63, 80,
    22,
    28,
    31,
    34,
    39,
    42,
    44,
    45,
    46,
    52,
    46,
    59,
    61,
    62,
    65,
    25,
    21,
    16,
    24,
    20,
    33,
    13,
    11,
    7,
    6,
]
j_params = {}
for sample, data in filtered_data.groupby("ROCK_FLAG"):
    data = data[~data["Sample"].isin(excluded_samples)]
    a, b = fit_j_curve(data["SWN"], data["J"])
    rmse = round(root_mean_squared_error(data["J"], a * data["SWN"] ** b), 4)
    j_params[sample] = (a, b, rmse)

In [ ]:
# Convert j_params dictionary to DataFrame
j_params_df = (
    pd.DataFrame.from_dict(j_params, orient="index", columns=["a", "b", "rmse"])
    .reset_index()
    .rename(columns={"index": "ROCK_FLAG"})
)
j_params_df.to_json(
    rf"data\04_project\{project_name}\outputs\j_params.json", orient="index"
)

# Merge filtered_data with j_params_df
merged_data = j_params_df.merge(
    filtered_data[["Sample", "ROCK_FLAG"]].drop_duplicates(), how="left", on="ROCK_FLAG"
)

# Group by ROCK_FLAG and sort by the first value of the parameters
sorted_samples = (
    merged_data.groupby("ROCK_FLAG")
    .apply(lambda x: x.sort_values(by="rmse"))
    .reset_index(drop=True)
)

# Select samples considering values of 'a'
selected_samples = {}
hist_rmse_values = set([0])
for rock_flag in sorted_samples["ROCK_FLAG"].unique():
    for _, row in (
        sorted_samples[sorted_samples["ROCK_FLAG"] == rock_flag]
        .sort_values(by="rmse")
        .iterrows()
    ):
        if row["rmse"] not in hist_rmse_values or all(
            row["rmse"] > val for val in hist_rmse_values
        ):
            selected_samples[rock_flag] = row
            hist_rmse_values.add(row["rmse"])
            break
        else:
            selected_samples[rock_flag] = row
selected_samples = pd.DataFrame(selected_samples).T
selected_samples.to_json(
    rf"data\04_project\{project_name}\outputs\rt_j_params.json", orient="index"
)

In [ ]:
import json

with open(rf"data\04_project\{project_name}\outputs\rt_j_params.json", "r") as file:
    mapped_fzi_params = json.load(file)
mapped_fzi_params

In [ ]:
from ipywidgets import interact, widgets

rt_widget = widgets.Dropdown(
    options=sorted(core_data["ROCK_FLAG"].unique()), description="Rock Type:"
)


@interact(rt=rt_widget)
def param(rt):
    str_rt = str(rt)
    a, b = mapped_fzi_params[str_rt]["a"], mapped_fzi_params[str_rt]["b"]
    data = core_data[core_data["ROCK_FLAG"] == rt]
    data = data[~data["SampleID"].isin(excluded_samples)]

    j_xplot(
        data["SWN"],
        data["J"],
        a=a,
        b=b,
        core_group=data["SampleID"],
        log_log=True,
        ylim=(0, 10),
        label=f"Rock Type {rt}: a:{a}, b:{b}",
    )

In [ ]:
from utils import plot_j_by_prt

plot_j_by_prt(core_data, mapped_fzi_params, ymax=10)

In [ ]:
import numpy as np

from quick_pp.utils import inv_power_law_func

# Plot mapped_fzi_params on the same j_xplot
for rock_flag, d in mapped_fzi_params.items():
    sample = d["Sample"]
    a, b = d["a"], d["b"]
    csw = np.geomspace(0.01, 1.0, 50)
    plt.plot(
        csw,
        inv_power_law_func(csw, a, b),
        label=f"RRT{rock_flag}, Sample {sample}, a:{a}, b:{b}",
        linestyle="dashed",
    )
plt.title("J Curve vs SW")
plt.xlabel("SW (v/v)")
plt.ylabel("J (unitless)")
plt.xlim(0, 1)
plt.ylim(0, 10)
plt.legend()

In [ ]:
STOP

***
## Log Derived Water Saturation

In [ ]:
from ipywidgets import widgets, interact

from quick_pp.saturation import pickett_plot

focused_data = df.copy()

wells = widgets.SelectMultiple(
    options=["All"] + list(focused_data["WELL_NAME"].unique()),
    value=["All"],
    description="Wells:",
)
m = widgets.FloatSlider(value=2, min=1, max=5, step=0.1, readout_format=".1f")
min_rw = widgets.FloatSlider(
    value=0.01, min=0.001, max=1.0, step=0.001, readout_format=".3f"
)
min_depth = widgets.FloatSlider(
    value=focused_data.DEPTH.min(),
    min=focused_data.DEPTH.min(),
    max=focused_data.DEPTH.max() - 10,
    step=0.1,
    readout_format=".1f",
)
max_depth = widgets.FloatSlider(
    value=focused_data.DEPTH.max(),
    min=focused_data.DEPTH.min() + 10,
    max=focused_data.DEPTH.max(),
    step=0.1,
    readout_format=".1f",
)


@interact(wells=wells, m=m, min_rw=min_rw, min_depth=min_depth, max_depth=max_depth)
def param(wells, m, min_rw, min_depth, max_depth):
    if "All" in wells:
        data = focused_data[
            (focused_data.DEPTH >= min_depth) & (focused_data.DEPTH <= max_depth)
        ]
    else:
        data = focused_data[
            (focused_data.WELL_NAME.isin(wells))
            & (focused_data.DEPTH >= min_depth)
            & (focused_data.DEPTH <= max_depth)
        ]
    pickett_plot(
        data["RT"],
        data["PHIT"],
        m=m,
        min_rw=min_rw,
        title=f"Pickett Plot for {wells[0]}",
    )

In [ ]:
import numpy as np
from matplotlib import ticker as mticker

from quick_pp.saturation import *
from quick_pp.porosity import *

water_salinity = 100e3
m = 2

temp_grad = estimate_temperature_gradient(well_data["TVD"], "metric")
rw = estimate_rw_temperature_salinity(temp_grad, water_salinity)
b = estimate_b_waxman_smits(temp_grad, rw)
qv = estimate_qv(well_data.VCLAY, well_data.PHIT, cec_clay=0.1)
phit_shale = estimate_shale_porosity(well_data.NPHI, well_data.PHIT)
rt_shale = estimate_rt_shale(well_data.RT, well_data.VCLAY)
qvn = estimate_qvn(well_data.VCLAY, well_data.PHIT, phit_shale)

swt_ws = waxman_smits_saturation(well_data["RT"], rw, well_data.PHIE, B=b, Qv=qvn, m=m)
swt_nws = normalized_waxman_smits_saturation(
    well_data.RT, rw, well_data.PHIT, well_data.VCLAY, phit_shale, rt_shale=8, m=m
)

swt_archie = archie_saturation(well_data.RT, rw, well_data.PHIT, m=m)

fig, axes = plt.subplots(3, 1, figsize=(15, 5), sharex=True)
axes[0].plot(well_data["DEPTH"], swt_nws, label="SWT Normalized WS")
axes[0].plot(well_data["DEPTH"], swt_ws, label="SWT WS")
axes[0].plot(well_data["DEPTH"], swt_archie, label="SWT Archie")
axes[0].plot(well_data["DEPTH"], np.ones(len(well_data)), color="black", linestyle="--")
axes[0].set_ylim(0, 2)
axes[0].legend()

axes[1].plot(well_data["DEPTH"], rw, label="RW")
axes[1].set_yscale("log")
axes[1].yaxis.set_minor_formatter(mticker.ScalarFormatter())
axes[1].legend()

axes[2].plot(well_data["DEPTH"], temp_grad, label="Temperature")
axes[2].legend()

fig.tight_layout()

***
# Plot the results

In [ ]:
from quick_pp.plotter.plotter import plotly_log
from quick_pp.core_calibration import *

# Plot individual results
well_data["SWT"] = swt_ws
fig = plotly_log(well_data, well_name=well_name, depth_uom="m")
fig.show(config=dict(scrollZoom=True))

# Apply to all

In [ ]:
from tqdm import tqdm

from quick_pp.database.objects import Project
from quick_pp.database.db_connector import DBConnector
from quick_pp.saturation import *
from quick_pp.porosity import *

db_conn = DBConnector()
project_name = "30-7a"
with db_conn.get_session() as db_session:
    project = Project(db_session, name=project_name)
    df = project.get_all_data()

water_salinity = 100e3
m = 2

for well_name, plot_data in tqdm(df.groupby("WELL_NAME")):
    tqdm.write(f"Processing {well_name}: {len(plot_data)} rows")

    temp_grad = estimate_temperature_gradient(plot_data["TVD"], "metric")
    rw = estimate_rw_temperature_salinity(temp_grad, water_salinity)
    b = estimate_b_waxman_smits(temp_grad, rw)
    phit_shale = estimate_shale_porosity(plot_data.NPHI, plot_data.PHIT)
    qvn = estimate_qvn(plot_data.VCLAY, plot_data.PHIT, phit_shale)
    swt = waxman_smits_saturation(plot_data["RT"], rw, plot_data.PHIE, B=b, Qv=qvn, m=m)

    plot_data["SWT"] = swt.clip(0, 1)

    # Save result to database
    with db_conn.get_session() as db_session:
        project = Project(db_session, name=project_name)
        project.update_data(plot_data)
        project.save()